# 11 — Final Results Visualisation

Presentation-ready summary of xG model performance and season-level impact.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from pathlib import Path
from IPython.display import display, Image

DATA_DIR = Path("../data")
TABLE_DIR = Path("../outputs/tables")
FIG_DIR = Path("../outputs/figures")
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


def assert_required_files(files: list[Path], label: str = "source") -> None:
    """Assert all files in the list exist, with a clear error message."""
    for f in files:
        assert f.exists(), f"Missing {label} file: {f.name}"


def assert_required_columns(
    df: pd.DataFrame, cols: list[str], name: str,
) -> None:
    """Assert all required columns are present in the DataFrame."""
    missing = set(cols) - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"


def save_png(fig: Figure, path: Path) -> None:
    """Save figure with standard conventions and close."""
    fig.tight_layout()
    fig.savefig(str(path), dpi=150, bbox_inches="tight")
    plt.close(fig)


METRIC_LABELS = {
    "roc_auc": "AUC",
    "log_loss": "Log Loss",
    "brier_score": "Brier Score",
}


def plot_metric_bar(
    df: pd.DataFrame,
    metric: str,
    title: str,
    path: Path,
    model_col: str = "model",
    colors: dict | None = None,
) -> None:
    """Horizontal bar chart for a single metric, fixed model order."""
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(
        df[model_col],
        df[metric],
        color=[colors[m] for m in df[model_col]] if colors else None,
    )
    ax.set_xlabel(METRIC_LABELS.get(metric, metric))
    ax.set_title(title)
    ax.invert_yaxis()
    for bar, val in zip(bars, df[metric]):
        ax.text(
            val, bar.get_y() + bar.get_height() / 2,
            f" {val:.4f}", va="center", fontsize=9,
        )
    save_png(fig, path)


print("Setup OK")

In [ ]:
MODEL_LABELS = {
    "baseline": "Baseline",
    "skill_aware": "Skill-aware",
    "form_aware": "Form-aware",
    "combined": "Combined",
}
MODEL_ORDER = list(MODEL_LABELS.values())
MODEL_COLORS = {
    "Baseline": "#1f77b4",
    "Skill-aware": "#ff8011",
    "Form-aware": "#2ca02c",
    "Combined": "#d62728",
}

MANDATORY_FILES = [
    DATA_DIR / "wyscout_xg_model_comparison_metrics.csv",
    DATA_DIR / "wyscout_xg_season_table.csv",
    DATA_DIR / "wyscout_xg_model_roc_comparison.png",
    DATA_DIR / "wyscout_xg_model_calibration_comparison.png",
    DATA_DIR / "wyscout_xg_season_points_comparison.png",
]
assert_required_files(MANDATORY_FILES)

COEFF_FILE = DATA_DIR / "wyscout_xg_model_coefficients.csv"
has_coefficients = COEFF_FILE.exists()

print(f"5 mandatory source files OK")
print(f"Coefficients file: {'found' if has_coefficients else 'not found (optional)'}")

## Model Performance

In [ ]:
metrics: pd.DataFrame = pd.read_csv(
    DATA_DIR / "wyscout_xg_model_comparison_metrics.csv",
)
assert_required_columns(
    metrics, ["model_name", "roc_auc", "log_loss", "brier_score"], "metrics",
)
assert set(metrics["model_name"]) == {
    "baseline", "skill_aware", "form_aware", "combined",
}, f"Unexpected model set: {set(metrics['model_name'])}"
assert len(metrics) == 4, f"Expected 4 model rows, got {len(metrics)}"

# Select and label
metrics_plot: pd.DataFrame = (
    metrics[["model_name", "roc_auc", "log_loss", "brier_score"]]
    .copy()
)
metrics_plot["model"] = metrics_plot["model_name"].map(MODEL_LABELS)
metrics_plot["model"] = pd.Categorical(
    metrics_plot["model"], categories=MODEL_ORDER, ordered=True,
)
metrics_plot = metrics_plot.sort_values("model").reset_index(drop=True)

# Rounded copy for display/export
metrics_display: pd.DataFrame = metrics_plot.copy()
for col in ["roc_auc", "log_loss", "brier_score"]:
    metrics_display[col] = metrics_display[col].round(4)

display(metrics_display[["model", "roc_auc", "log_loss", "brier_score"]])
metrics_display[["model", "roc_auc", "log_loss", "brier_score"]].to_csv(
    TABLE_DIR / "final_model_metrics_summary.csv", index=False,
)
print(f"Saved: {TABLE_DIR / 'final_model_metrics_summary.csv'}")

In [ ]:
plot_metric_bar(
    metrics_plot, "roc_auc", "ROC AUC by Model",
    FIG_DIR / "auc_comparison.png", colors=MODEL_COLORS,
)
print("Saved: auc_comparison.png")

In [ ]:
plot_metric_bar(
    metrics_plot, "log_loss", "Log Loss by Model",
    FIG_DIR / "logloss_comparison.png", colors=MODEL_COLORS,
)
print("Saved: logloss_comparison.png")

In [ ]:
plot_metric_bar(
    metrics_plot, "brier_score", "Brier Score by Model",
    FIG_DIR / "brier_comparison.png", colors=MODEL_COLORS,
)
print("Saved: brier_comparison.png")

In [ ]:
display(Image(filename=str(DATA_DIR / "wyscout_xg_model_roc_comparison.png")))
display(Image(filename=str(DATA_DIR / "wyscout_xg_model_calibration_comparison.png")))

## League Table Comparison

In [ ]:
season: pd.DataFrame = pd.read_csv(
    DATA_DIR / "wyscout_xg_season_table.csv", index_col=0,
).reset_index(drop=True)

SEASON_COLS = [
    "team_name", "actual_pts", "goal_diff", "goals_for",
    "baseline_xg_pts", "xg_diff_baseline", "xg_for_baseline",
    "combined_xg_pts", "xg_diff_combined", "xg_for_combined",
]
assert_required_columns(season, SEASON_COLS, "season")
assert len(season) == 20, f"Expected 20 teams, got {len(season)}"
assert season["team_name"].nunique() == 20, (
    f"Expected 20 unique teams, got {season['team_name'].nunique()}"
)


def compute_rank(
    df: pd.DataFrame,
    pts_col: str,
    diff_col: str,
    for_col: str,
) -> pd.Series:
    """Compute 1-based rank with deterministic tiebreakers."""
    sorted_df = df.sort_values(
        [pts_col, diff_col, for_col, "team_name"],
        ascending=[False, False, False, True],
    )
    rank_map = {name: i + 1 for i, name in enumerate(sorted_df["team_name"])}
    return df["team_name"].map(rank_map)


comparison: pd.DataFrame = season[["team_name", "actual_pts", "baseline_xg_pts", "combined_xg_pts"]].copy()

comparison["actual_rank"] = compute_rank(
    season, "actual_pts", "goal_diff", "goals_for",
)
comparison["baseline_rank"] = compute_rank(
    season, "baseline_xg_pts", "xg_diff_baseline", "xg_for_baseline",
)
comparison["combined_rank"] = compute_rank(
    season, "combined_xg_pts", "xg_diff_combined", "xg_for_combined",
)

# Signed differences
comparison["baseline_points_diff"] = comparison["baseline_xg_pts"] - comparison["actual_pts"]
comparison["combined_points_diff"] = comparison["combined_xg_pts"] - comparison["actual_pts"]
comparison["baseline_rank_diff"] = comparison["baseline_rank"] - comparison["actual_rank"]
comparison["combined_rank_diff"] = comparison["combined_rank"] - comparison["actual_rank"]
comparison["baseline_rank_abs_error"] = comparison["baseline_rank_diff"].abs()
comparison["combined_rank_abs_error"] = comparison["combined_rank_diff"].abs()

# Rank integrity checks
expected_ranks = set(range(1, 21))
for col in ["actual_rank", "baseline_rank", "combined_rank"]:
    assert set(comparison[col]) == expected_ranks, (
        f"{col} ranks not 1..20: {sorted(comparison[col])}"
    )
    assert comparison[col].nunique() == 20, f"{col} has duplicate ranks"

comparison = comparison.sort_values("actual_rank").reset_index(drop=True)
display(comparison)

comparison.to_csv(TABLE_DIR / "final_league_table_comparison.csv", index=False)
print(f"Saved: {TABLE_DIR / 'final_league_table_comparison.csv'}")

In [ ]:
display(Image(filename=str(DATA_DIR / "wyscout_xg_season_points_comparison.png")))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    comparison["actual_pts"], comparison["baseline_xg_pts"],
    s=60, zorder=3, color=MODEL_COLORS["Baseline"],
)

lo = min(comparison["actual_pts"].min(), comparison["baseline_xg_pts"].min()) - 3
hi = max(comparison["actual_pts"].max(), comparison["baseline_xg_pts"].max()) + 3
ax.plot([lo, hi], [lo, hi], ls="--", color="grey", alpha=0.6, zorder=1)

for _, row in comparison.iterrows():
    ax.annotate(
        row["team_name"],
        (row["actual_pts"], row["baseline_xg_pts"]),
        fontsize=7, ha="left", va="bottom",
        xytext=(4, 4), textcoords="offset points",
    )

ax.set_xlabel("Actual Points")
ax.set_ylabel("Baseline xG Points")
ax.set_title("2017/18 EPL: Actual vs Baseline xG Points")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

save_png(fig, FIG_DIR / "actual_vs_baseline_xpts.png")
print("Saved: actual_vs_baseline_xpts.png")

In [ ]:
comp_sorted = comparison.sort_values("actual_rank")
teams = comp_sorted["team_name"]
y = np.arange(len(teams))
bar_height = 0.35

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    y - bar_height / 2, comp_sorted["baseline_rank_diff"],
    bar_height, label="Baseline", color=MODEL_COLORS["Baseline"],
)
ax.barh(
    y + bar_height / 2, comp_sorted["combined_rank_diff"],
    bar_height, label="Combined", color=MODEL_COLORS["Combined"],
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(teams)
ax.invert_yaxis()
ax.set_xlabel("Rank Difference (model \u2212 actual)")
ax.set_title("Rank Difference from Actual (xG Models)")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)

save_png(fig, FIG_DIR / "rank_difference_comparison.png")
print("Saved: rank_difference_comparison.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    y - bar_height / 2, comp_sorted["baseline_points_diff"],
    bar_height, label="Baseline", color=MODEL_COLORS["Baseline"],
)
ax.barh(
    y + bar_height / 2, comp_sorted["combined_points_diff"],
    bar_height, label="Combined", color=MODEL_COLORS["Combined"],
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(teams)
ax.invert_yaxis()
ax.set_xlabel("Points Difference (model \u2212 actual)")
ax.set_title("Points Difference from Actual (xG Models)")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)

save_png(fig, FIG_DIR / "points_difference_comparison.png")
print("Saved: points_difference_comparison.png")

In [ ]:
if has_coefficients:
    coeffs: pd.DataFrame = pd.read_csv(COEFF_FILE)
    assert_required_columns(
        coeffs, ["model_name", "feature", "coefficient", "abs_coefficient"],
        "coefficients",
    )
    combined_coeffs = coeffs[coeffs["model_name"] == "combined"].copy()
    assert len(combined_coeffs) > 0, "No combined model coefficients found"

    top15 = combined_coeffs.nlargest(15, "abs_coefficient")

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["#d62728" if v > 0 else "#1f77b4" for v in top15["coefficient"]]
    ax.barh(top15["feature"], top15["coefficient"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("Coefficient")
    ax.set_title("Combined Model \u2014 Top 15 Features by |Coefficient|")
    ax.grid(True, axis="x", alpha=0.3)

    save_png(fig, FIG_DIR / "feature_importance_combined_top15.png")
    print("Saved: feature_importance_combined_top15.png")
else:
    print("Coefficients file not found \u2014 skipping feature importance chart")

In [ ]:
print("=== Reused source artefacts (displayed inline) ===")
for f in [
    "wyscout_xg_model_roc_comparison.png",
    "wyscout_xg_model_calibration_comparison.png",
    "wyscout_xg_season_points_comparison.png",
]:
    print(f"  data/{f}")

print("\n=== New tables written by Notebook 11 ===")
new_tables = [
    TABLE_DIR / "final_model_metrics_summary.csv",
    TABLE_DIR / "final_league_table_comparison.csv",
]
for t in new_tables:
    assert t.exists(), f"Missing output: {t.name}"
    print(f"  {t}")

print("\n=== New figures written by Notebook 11 ===")
new_figures = [
    FIG_DIR / "auc_comparison.png",
    FIG_DIR / "logloss_comparison.png",
    FIG_DIR / "brier_comparison.png",
    FIG_DIR / "actual_vs_baseline_xpts.png",
    FIG_DIR / "rank_difference_comparison.png",
    FIG_DIR / "points_difference_comparison.png",
]
if has_coefficients:
    new_figures.append(FIG_DIR / "feature_importance_combined_top15.png")
for f in new_figures:
    assert f.exists(), f"Missing output: {f.name}"
    print(f"  {f}")

print(f"\nAll outputs verified ({len(new_tables)} tables, {len(new_figures)} figures)")